In [ ]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms.functional as TF

class SRImplicitDataset(Dataset):
    def __init__(self, img_dir, max_images=100):
        self.files = sorted([
            os.path.join(img_dir, f)
            for f in os.listdir(img_dir)
            if f.endswith(".png") or f.endswith(".jpg")
        ])[:max_images]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        hr = TF.to_tensor(img)

        h, w = hr.shape[1:]

        # HARD CLAMP HR (NO SKIP)
        h = max(2, h)
        w = max(2, w)
        hr = TF.resize(hr, (h, w), antialias=True)

        scale = random.uniform(1.5, 4.0)

        # SAFE LR (MIN = 2)
        lr_h = max(2, int(h / scale))
        lr_w = max(2, int(w / scale))

        lr = TF.resize(hr, (lr_h, lr_w), antialias=True)

        return lr, hr

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SRNOInspired(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, hidden, 3, padding=1),
            nn.ReLU()
        )

        self.mlp = nn.Sequential(
            nn.Linear(hidden + 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 3)
        )

    def forward(self, lr, out_h, out_w):
        B, _, H_lr, W_lr = lr.shape

        # SAFE INPUT SIZE
        if H_lr < 2 or W_lr < 2:
            lr = F.interpolate(lr, size=(max(2, H_lr), max(2, W_lr)), mode="bilinear", align_corners=False)

        feat = self.encoder(lr)

        B, C, H, W = feat.shape

        # SAFE FEATURE SIZE
        if H < 2 or W < 2:
            feat = F.interpolate(feat, size=(max(2, H), max(2, W)), mode="bilinear", align_corners=False)
            B, C, H, W = feat.shape

        # SAFE OUTPUT SIZE
        out_h = max(2, out_h)
        out_w = max(2, out_w)

        # coordinate grid
        y = torch.linspace(-1, 1, out_h, device=lr.device)
        x = torch.linspace(-1, 1, out_w, device=lr.device)
        yy, xx = torch.meshgrid(y, x, indexing="ij")

        grid = torch.stack((xx, yy), dim=-1)
        grid = grid.unsqueeze(0).repeat(B, 1, 1, 1)

        # safe grid_sample
        feat_up = F.grid_sample(
            feat,
            grid,
            mode='bilinear',
            align_corners=True,
            padding_mode='border'
        )

        feat_flat = feat_up.permute(0, 2, 3, 1).reshape(-1, C)
        coords_flat = grid.reshape(-1, 2)

        inp = torch.cat([feat_flat, coords_flat], dim=1)
        out = self.mlp(inp)

        out = out.view(B, out_h, out_w, 3).permute(0, 3, 1, 2)

        return out